In [ ]:
import os
import random
import numpy as np
import torch
import optuna
import pandas as pd

from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback

from agent import FloodWarningEnv

# --- Curriculum settings ---
SEVERE_PROB_START = 0.9   # start with 90% severe episodes
SEVERE_PROB_END   = 0.0   # decay to 0% by end of training
CURRICULUM_END_FRAC = 0.5 # point which curriculum finishes through training

class CurriculumCallback(BaseCallback):
    def __init__(self, envs, total_steps, start_prob, end_prob, end_frac, verbose=0):
        super().__init__(verbose)
        self.envs = envs
        self.total_steps = total_steps
        self.start_prob = start_prob
        self.end_prob = end_prob
        self.curriculum_steps = int(total_steps * end_frac)

    def _on_step(self) -> bool:
        progress = min(self.num_timesteps / self.curriculum_steps, 1.0)
        severe_prob = self.start_prob + progress * (self.end_prob - self.start_prob)

        for env in self.envs:
            env.set_severe_prob(severe_prob)

        return True


def make_env(severe_prob):
    return lambda: FloodWarningEnv(severe_prob=severe_prob)

In [ ]:
SEED = 1
N_TRIALS_PER_RUN = 50
TRAIN_STEPS = 300_000
N_EVAL_EPISODES = 50

STUDY_NAME = "flood_warning_tuning_lstm"
OUTPUT_DIR = "./tuning/tuning_lstm/"
DB_PATH = "sqlite:///./tuning/tuning_lstm/optuna_study_lstm.db"

os.makedirs(OUTPUT_DIR, exist_ok=True)

def objective(trial):
    # --- Hyperparameters ---
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    n_steps = trial.suggest_categorical("n_steps", [256, 512, 1024])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    n_epochs = trial.suggest_int("n_epochs", 5, 20)
    ent_coef = trial.suggest_float("ent_coef", 1e-4, 0.1, log=True)

    # --- Seeds ---
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # --- Environments ---
    env = make_vec_env(
        make_env(severe_prob=SEVERE_PROB_START),
        n_envs=8,
        seed=SEED
    )

    eval_env = make_vec_env(
        make_env(severe_prob=0.0),  # no curriculum for evaluation
        n_envs=1,
        seed=SEED
    )

    # Get underlying envs for curriculum updates
    train_envs = [env.envs[i] for i in range(8)]

    # --- Model ---
    model = RecurrentPPO(
        "MlpLstmPolicy",
        env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        n_epochs=n_epochs,
        ent_coef=ent_coef,
        gamma=0.99,
        verbose=0,
        seed=SEED,
    )

    # --- Callbacks ---
    curriculum_callback = CurriculumCallback(
        envs=train_envs,
        total_steps=TRAIN_STEPS,
        start_prob=SEVERE_PROB_START,
        end_prob=SEVERE_PROB_END,
        end_frac=CURRICULUM_END_FRAC,
        verbose=0,
    )

    eval_callback = EvalCallback(
        eval_env,
        eval_freq=25_000,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
        verbose=0,
    )

    # --- Training ---
    model.learn(
        total_timesteps=TRAIN_STEPS,
        callback=[curriculum_callback, eval_callback]
    )

    mean_reward = eval_callback.last_mean_reward

    env.close()
    eval_env.close()

    return mean_reward


if __name__ == "__main__":
    study = optuna.create_study(
        direction="maximize",
        storage=DB_PATH,
        study_name=STUDY_NAME,
        load_if_exists=True
    )

    study.optimize(objective, n_trials=N_TRIALS_PER_RUN)

    print("Best trial:")
    print(f"  Mean reward: {study.best_trial.value:.4f}")
    print(f"  Params: {study.best_trial.params}")

    study.trials_dataframe().to_csv(
        os.path.join(OUTPUT_DIR, "study_results.csv"),
        index=False
    )

    print("Study results saved to " + os.path.join(OUTPUT_DIR, "study_results.csv"))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

df = pd.read_csv("./tuning/tuning_lstm/study_results.csv")
df = df[df["state"] == "COMPLETE"].copy()
df["rank"] = df["value"].rank(ascending=False)

params = [
    ("params_learning_rate", "Learning Rate", True),   # (col, label, log_scale)
    ("params_n_steps",       "N Steps",        False),
    ("params_batch_size",    "Batch Size",     False),
    ("params_n_epochs",      "N Epochs",       False),
    ("params_ent_coef",      "Entropy Coef",   True),
    ("params_false_weight",  "False Weight",   False),
    ("params_missed_weight", "Missed Weight",  False),
    ("params_jump_weight",   "Jump Weight",    False),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

norm = plt.Normalize(df["value"].min(), df["value"].max())
cmap = cm.viridis

for ax, (col, label, log_scale) in zip(axes, params):
    sc = ax.scatter(df[col], df["value"],
                    c=df["value"], cmap=cmap, norm=norm,
                    alpha=0.6, edgecolors="none", s=30)
    ax.set_xlabel(label)
    ax.set_ylabel("Mean Reward")
    ax.set_title(label)
    if log_scale:
        ax.set_xscale("log")

fig.colorbar(sc, ax=axes, label="Mean Reward", fraction=0.02, pad=0.04)
plt.suptitle("Hyperparameter Sensitivity - 200 runs - ", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("sensitivity_scatter.png", dpi=150, bbox_inches="tight")

In [ ]:
df = pd.read_csv("tuning/tuning_lstm/study_results.csv")
best_row = df.loc[df["value"].idxmax()]
print(best_row)

In [ ]:
SEED = 1
TRAIN_STEPS = 10_000_000
N_EVAL_EPISODES = 50

OUTPUT_DIR = "./models/lstm/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Hyperparameters ---
LEARNING_RATE = 0
N_STEPS = 0
BATCH_SIZE = 0
N_EPOCHS = 0
ENT_COEF = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Training envs start with severe_prob=SEVERE_PROB_START
env = make_vec_env(make_env(severe_prob=SEVERE_PROB_START), n_envs=8, seed=SEED)

# Eval env always uses natural distribution (no curriculum)
eval_env = make_vec_env(make_env(severe_prob=0.0), n_envs=1, seed=SEED)

# Get references to underlying envs for curriculum callback to update
train_envs = [env.envs[i] for i in range(8)]

# Agent
model = RecurrentPPO(
    "MlpLstmPolicy",
    env,
    learning_rate=LEARNING_RATE,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    ent_coef=ENT_COEF,
    gamma=0.99,
    verbose=1,
    seed=SEED,
)

# Callbacks
curriculum_callback = CurriculumCallback(
    envs=train_envs,
    total_steps=TRAIN_STEPS,
    start_prob=SEVERE_PROB_START,
    end_prob=SEVERE_PROB_END,
    end_frac=CURRICULUM_END_FRAC,
    verbose=1,
)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=OUTPUT_DIR,
    eval_freq=25_000,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True,
    verbose=1,
)

model.learn(
    total_timesteps=TRAIN_STEPS,
    callback=[curriculum_callback, eval_callback]
)

model.save(os.path.join(OUTPUT_DIR, "final_model"))

print(f"Training complete.")
print(f"Best mean reward: {eval_callback.best_mean_reward:.4f}")
print(f"Final mean reward: {eval_callback.last_mean_reward:.4f}")
print(f"Model saved to {OUTPUT_DIR}")

env.close()
eval_env.close()